# RI-JK RHF Hessian：CP-HF 分解 (2) 方程求解

In [1]:
from pyscf import gto, scf, lib, df, hessian
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_r_hf_decomp.npz")["de_cphf"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
occ_energy = mo_energy[mo_occ > 0]

In [7]:
eocc = mo_energy[mo_occ > 0]
evir = mo_energy[mo_occ == 0]
mvir = mo_coeff[:, mo_occ == 0]

## Overview

In [8]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")
    
    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1] 
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [9]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
mo1, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1 = np.array(mo1)
mo_e1 = np.array(mo_e1)

In [10]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])

## Response Function

### original vresp 

CP-HF/KS need response function (instead of usual fock/veff). However, for Hartree-Fock problem, response (F) is identical to potential (V).

This part need to be expanded for KS counterpart.

In [11]:
vresp = mf.gen_response()

In [12]:
dm_rand = np.random.rand(nao, nao)
dm_rand += dm_rand.T
np.allclose(vresp(dm_rand), mf.get_veff(dm=dm_rand))

True

In [13]:
np.allclose((
    + np.einsum("uvP, PQ, klQ, kl -> uv", int3c2e, int2c2e_inv, int3c2e, dm_rand)
    - 0.5 * np.einsum("uvP, PQ, klQ, vl -> uk", int3c2e, int2c2e_inv, int3c2e, dm_rand)
), vresp(dm_rand))

True

### original vind

In [14]:
mo1_rand = np.random.rand(3, nmo, nocc)

In [15]:
dm1_rand = mo_coeff @ (mo1_rand * 2) @ mocc.T
dm1_rand += dm1_rand.transpose(0, 2, 1)
np.allclose(
    + np.einsum("uvP, PQ, klQ, Akl, up, vi -> Api", int3c2e, int2c2e_inv, int3c2e, dm1_rand, mo_coeff, mocc)
    - 0.5 * np.einsum("uvP, PQ, klQ, Avl, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, dm1_rand, mo_coeff, mocc)
,
    hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand)
)

True

### vind decomposition

Note for better optimization, the following code may be better optimized than the original vind (since there is another one AO indcies can be pre-contracted to occupied orbital indices).

In [16]:
np.allclose(
    + 4 * np.einsum("uvP, PQ, klQ, Aqj, kq, lj, up, vi -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
    - np.einsum("uvP, PQ, klQ, Aqj, vq, lj, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
    - np.einsum("uvP, PQ, klQ, Aqj, lq, vj, up, ki -> Api", int3c2e, int2c2e_inv, int3c2e, mo1_rand, mo_coeff, mocc, mo_coeff, mocc)
,
    hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand)
)

True

In [17]:
mo1_rand_half_trans = mo_coeff @ mo1_rand
t = (
    + 4 * np.einsum("uvP, PQ, klQ, Akj, lj, vi -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
    - 1 * np.einsum("uvP, PQ, klQ, Avj, lj, ki -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
    - 1 * np.einsum("uvP, PQ, klQ, Akj, vj, li -> Aui", int3c2e, int2c2e_inv, int3c2e, mo1_rand_half_trans, mocc, mocc)
)
r = np.einsum("Aui, up -> Api", t, mo_coeff)
np.allclose(r, hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)(mo1_rand))

True

### CP-HF recover

In [18]:
# CP-HF requires to pre-flatten the tensors to (comp, nao, nao)
s1ao = np.array([ovlp_deriv1_generator(mol)(A) for A in range(natm)])
f1ao = np.array(mf_hess.make_h1(mo_coeff, mo_occ))
s1ao = s1ao.reshape(-1, nao, nao)
f1ao = f1ao.reshape(-1, nao, nao)

In [19]:
level_shift = 0
e_ai = 1 / (evir[:, None] - eocc[None, :] + level_shift)

In [20]:
f1mo = mo_coeff.T @ f1ao @ mocc
s1mo = mo_coeff.T @ s1ao @ mocc
hsmo = f1mo - s1mo * eocc

mo1_base = np.zeros_like(hsmo)
mo1_base[:, nocc:] = -hsmo[:, nocc:] * e_ai
mo1_base[:, :nocc] = -s1mo[:, :nocc] * 0.5

fvind = hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)

def vind_vo(mo1):
    mo1 = mo1.reshape(-1, nmo, nocc)
    v = fvind(mo1).reshape(-1, nmo, nocc)
    if level_shift != 0:
        v -= mo1 * level_shift
    v[:, nocc:, :] *= e_ai
    v[:, :nocc, :] = 0
    return v.reshape(-1, nmo * nocc)

mo1_ = lib.krylov(vind_vo, mo1_base.reshape(-1, nmo * nocc))
mo1_ = mo1_.reshape(-1, nmo, nocc)
mo1_[:, :nocc] = mo1_base[:, :nocc]

hsmo += fvind(mo1_)
mo1_[:, nocc:] = hsmo[:, nocc:] / (eocc - evir[:, None])

mo_e1_ = hsmo[:, :nocc, :]
mo_e1_ += mo1_[:, :nocc] * (eocc[:, None] - eocc)

mo1_ = mo1_.reshape((natm, 3, nmo, nocc))
mo_e1_ = mo_e1_.reshape((natm, 3, nocc, nocc))